# LOSO Cross-Validation: RBF + Periodic Kernel

This notebook adds a **locally periodic kernel** to the baseline RBF kernel, following
[Patel et al. (2022, AAAI)](https://cdn.aaai.org/ojs/21467/21467-13-25480-1-2-20220628.pdf).

**Kernel structure:**
```
k(x, x') = ScaleKernel(RBF_ARD(all features))
          + ScaleKernel(Periodic(day_of_year) × RBF(day_of_year))
```

The locally periodic component (Periodic × RBF) captures annual cyclicity in PM2.5
while allowing the pattern amplitude to evolve. The RBF envelope prevents the kernel
from assuming exact repetition year-over-year.

**Scale considerations:** Patel et al. use hourly urban data from India. Our data is
daily observations over 2018–2019 in Montana, so the dominant periodicity is the
**annual cycle** (~365 days) rather than diurnal. We initialize the periodic kernel's
period to 365.25 days (converted to standardized units per fold) and let it optimize.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import torch
import gpytorch
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

from timing_utils import TimingLogger, Timer

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

timing_log = TimingLogger("../timings/gpr_timings.csv", experiment_name="periodic_loso_cv")
print("Timing will be logged to timings/gpr_timings.csv")

## 1. Load and Prepare Data

Same as baseline, plus a `day_of_year` feature for the periodic kernel.

In [ ]:
# Load datasets
pm_all = pd.read_csv("../../data/pm25_data_complete_2003-2021_nodups_051922.csv", low_memory=False)
pm_fixed = pd.read_csv('../../eda/pm25_locs_with_states.csv')

# Filter to Montana
mt_sites = pm_fixed[pm_fixed['state'] == 'MT'].copy()
mt_ll_ids = set(mt_sites['ll_id'].values)

# Parse date and filter to 2018-2019
pm_all['date'] = pd.to_datetime(pm_all['date'], format='%Y%m%d')
pm_all['year'] = pm_all['date'].dt.year
pm_mt = pm_all[(pm_all['ll_id'].isin(mt_ll_ids)) & (pm_all['year'].isin([2018, 2019]))].copy()

print(f"Montana 2018-2019: {len(pm_mt):,} observations from {pm_mt['ll_id'].nunique()} sites")

In [ ]:
# Define features
time_varying_features = ['aot', 'wind', 'hgt', 'cld', 'longwave', 'rh', 'tmax', 'cadI', 'smogP']
static_features = ['lat', 'lon', 'logpd2500g', 'minf_5000', 'sd50k',
                   'heavy_industrial_ind1', 'housing']

available_tv = [f for f in time_varying_features if f in pm_mt.columns]
available_static = [f for f in static_features if f in mt_sites.columns]

# Merge datasets
pm_mt_subset = pm_mt[['ll_id', 'date', 'pm25'] + available_tv].copy()
mt_static = mt_sites[['ll_id'] + available_static].copy()
df = pm_mt_subset.merge(mt_static, on='ll_id', how='left')

# Add day_of_year temporal feature for the periodic kernel
df['day_of_year'] = df['date'].dt.dayofyear

# Feature columns: weather + static + temporal
temporal_feature = 'day_of_year'
feature_cols = available_tv + available_static + [temporal_feature]
temporal_idx = feature_cols.index(temporal_feature)

# Clean data
df_clean = df.dropna(subset=feature_cols + ['pm25']).copy()
print(f"Clean data: {len(df_clean):,} observations")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Temporal feature '{temporal_feature}' at index {temporal_idx}")

In [ ]:
# Check observations per site
site_counts = df_clean.groupby('ll_id').size().sort_values(ascending=False)
print(f"Observations per site:")
print(f"  Min: {site_counts.min()}")
print(f"  Max: {site_counts.max()}")
print(f"  Mean: {site_counts.mean():.1f}")
print(f"  Median: {site_counts.median():.1f}")

# Show day_of_year distribution
print(f"\nday_of_year range: {df_clean['day_of_year'].min()} - {df_clean['day_of_year'].max()}")
print(f"day_of_year std: {df_clean['day_of_year'].std():.1f}")
print(f"Expected period in standardized units: {365.25 / df_clean['day_of_year'].std():.2f}")

## 2. Define GP Model with Periodic Kernel

In [ ]:
class PeriodicGPModel(gpytorch.models.ExactGP):
    """GP model with RBF + Locally Periodic kernel.

    Kernel = ScaleKernel(RBF_ARD(all features))
           + ScaleKernel(Periodic(temporal) * RBF(temporal))

    The locally periodic component (Periodic x RBF) captures annual
    cyclicity while allowing the pattern amplitude to evolve over time.
    Following Patel et al. (2022, AAAI).
    """

    def __init__(self, train_x, train_y, likelihood, n_features, temporal_idx, period_init=None):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()

        # Component 1: RBF with ARD on all features (same as baseline)
        self.rbf_kernel = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=n_features)
        )

        # Component 2: Locally periodic on temporal dimension
        self.temporal_periodic = gpytorch.kernels.PeriodicKernel(
            active_dims=torch.tensor([temporal_idx])
        )
        if period_init is not None:
            self.temporal_periodic.initialize(period_length=period_init)

        self.temporal_rbf = gpytorch.kernels.RBFKernel(
            active_dims=torch.tensor([temporal_idx])
        )

        self.periodic_kernel = gpytorch.kernels.ScaleKernel(
            self.temporal_periodic * self.temporal_rbf
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.rbf_kernel(x) + self.periodic_kernel(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [ ]:
def train_gp(train_x, train_y, n_features, temporal_idx, period_init, n_epochs=50, lr=0.1, verbose=False):
    """Train a GP model with periodic kernel and return it."""
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    model = PeriodicGPModel(train_x, train_y, likelihood, n_features,
                            temporal_idx, period_init).to(device)

    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        optimizer.step()

        if verbose and (i + 1) % 20 == 0:
            print(f"  Epoch {i+1}, Loss: {loss.item():.4f}")

    return model, likelihood

In [ ]:
def predict_gp(model, likelihood, test_x):
    """Make predictions with a trained GP."""
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(model(test_x))
        return pred.mean.cpu().numpy(), pred.variance.cpu().numpy()

## 3. Run LOSO Cross-Validation

In [ ]:
# Settings
MAX_TRAIN_SIZE = 2000
N_EPOCHS = 50

sites = df_clean['ll_id'].unique()
n_sites = len(sites)
print(f"Running LOSO CV across {n_sites} sites...")

In [ ]:
# Storage for results
all_predictions = []
all_actuals = []
all_variances = []
site_metrics = []
fold_timings = []
learned_periods = []  # Track what period the kernel learns

cv_start = time.perf_counter()

for i, held_out_site in enumerate(tqdm(sites, desc="LOSO CV")):
    fold_start = time.perf_counter()

    # Split data
    test_mask = df_clean['ll_id'] == held_out_site
    train_df = df_clean[~test_mask]
    test_df = df_clean[test_mask]

    if len(test_df) == 0:
        continue

    # Extract features and target
    X_train = train_df[feature_cols].values
    y_train = np.log(train_df['pm25'].values + 1)
    X_test = test_df[feature_cols].values
    y_test = np.log(test_df['pm25'].values + 1)

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Compute period in standardized units
    # Original annual period = 365.25 days
    # After standardization: period_std = 365.25 / std(day_of_year)
    period_init = 365.25 / scaler.scale_[temporal_idx]

    # Subsample training data if needed
    n_train_actual = len(X_train_scaled)
    if len(X_train_scaled) > MAX_TRAIN_SIZE:
        idx = np.random.choice(len(X_train_scaled), MAX_TRAIN_SIZE, replace=False)
        X_train_scaled = X_train_scaled[idx]
        y_train = y_train[idx]
        n_train_actual = MAX_TRAIN_SIZE

    # Convert to tensors
    train_x = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    train_y = torch.tensor(y_train, dtype=torch.float32).to(device)
    test_x = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

    # Train with timing
    train_start = time.perf_counter()
    model, likelihood = train_gp(train_x, train_y, len(feature_cols),
                                  temporal_idx, period_init, N_EPOCHS)
    train_time = time.perf_counter() - train_start

    # Predict with timing
    infer_start = time.perf_counter()
    pred_mean, pred_var = predict_gp(model, likelihood, test_x)
    infer_time = time.perf_counter() - infer_start

    fold_time = time.perf_counter() - fold_start

    # Record learned period (convert back to days)
    learned_period_std = model.temporal_periodic.period_length.item()
    learned_period_days = learned_period_std * scaler.scale_[temporal_idx]
    learned_periods.append({
        'fold': i,
        'site': held_out_site,
        'period_init_std': period_init,
        'period_learned_std': learned_period_std,
        'period_learned_days': learned_period_days
    })

    # Store predictions
    all_predictions.extend(pred_mean)
    all_actuals.extend(y_test)
    all_variances.extend(pred_var)

    # Calculate site-level metrics
    site_rmse = np.sqrt(np.mean((pred_mean - y_test)**2))
    site_mae = np.mean(np.abs(pred_mean - y_test))
    ss_res = np.sum((y_test - pred_mean)**2)
    ss_tot = np.sum((y_test - np.mean(y_test))**2)
    site_r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

    site_metrics.append({
        'site': held_out_site,
        'n_obs': len(test_df),
        'rmse_log': site_rmse,
        'mae_log': site_mae,
        'r2_log': site_r2
    })

    fold_timings.append({
        'fold': i,
        'site': held_out_site,
        'n_train': n_train_actual,
        'n_test': len(test_df),
        'train_time': train_time,
        'infer_time': infer_time,
        'fold_time': fold_time
    })

    # Clean up GPU memory
    del model, likelihood
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

cv_total_time = time.perf_counter() - cv_start

timing_log.log("periodic_loso_cv_total", cv_total_time, years="2018-2019",
               n_folds=n_sites, n_features=len(feature_cols), n_epochs=N_EPOCHS,
               kernel="RBF_ARD+Periodic")

print(f"\nLOSO CV complete!")
print(f"Total CV time: {cv_total_time:.1f}s ({cv_total_time/60:.1f} min)")

## 4. Aggregate Results

In [ ]:
# Convert to arrays
all_predictions = np.array(all_predictions)
all_actuals = np.array(all_actuals)
all_variances = np.array(all_variances)

# Overall metrics on log scale
rmse_log = np.sqrt(np.mean((all_predictions - all_actuals)**2))
mae_log = np.mean(np.abs(all_predictions - all_actuals))
ss_res = np.sum((all_actuals - all_predictions)**2)
ss_tot = np.sum((all_actuals - np.mean(all_actuals))**2)
r2_log = 1 - (ss_res / ss_tot)

print("=" * 50)
print("LOSO CV Results (Periodic + RBF) - Log Scale")
print("=" * 50)
print(f"Total predictions: {len(all_predictions):,}")
print(f"RMSE: {rmse_log:.4f}")
print(f"MAE:  {mae_log:.4f}")
print(f"R\u00b2:   {r2_log:.4f}")

In [ ]:
# Transform to original scale
pred_pm25 = np.exp(all_predictions) - 1
actual_pm25 = np.exp(all_actuals) - 1

rmse_orig = np.sqrt(np.mean((pred_pm25 - actual_pm25)**2))
mae_orig = np.mean(np.abs(pred_pm25 - actual_pm25))
ss_res_orig = np.sum((actual_pm25 - pred_pm25)**2)
ss_tot_orig = np.sum((actual_pm25 - np.mean(actual_pm25))**2)
r2_orig = 1 - (ss_res_orig / ss_tot_orig)

print("\n" + "=" * 50)
print("LOSO CV Results (Periodic + RBF) - Original Scale (\u03bcg/m\u00b3)")
print("=" * 50)
print(f"RMSE: {rmse_orig:.2f}")
print(f"MAE:  {mae_orig:.2f}")
print(f"R\u00b2:   {r2_orig:.4f}")

In [ ]:
# Site-level metrics summary
metrics_df = pd.DataFrame(site_metrics)
print("\nSite-level metrics summary:")
print(metrics_df[['rmse_log', 'mae_log', 'r2_log']].describe())

## 5. Visualizations

In [ ]:
# Predictions vs Actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Log scale
axes[0].scatter(all_actuals, all_predictions, alpha=0.2, s=5)
axes[0].plot([all_actuals.min(), all_actuals.max()],
             [all_actuals.min(), all_actuals.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual log(PM2.5 + 1)')
axes[0].set_ylabel('Predicted log(PM2.5 + 1)')
axes[0].set_title(f'Periodic+RBF LOSO - Log Scale (R\u00b2 = {r2_log:.3f})')
axes[0].grid(True, alpha=0.3)

# Original scale
axes[1].scatter(actual_pm25, pred_pm25, alpha=0.2, s=5)
max_val = np.percentile(np.concatenate([actual_pm25, pred_pm25]), 99)
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=2)
axes[1].set_xlabel('Actual PM2.5 (\u03bcg/m\u00b3)')
axes[1].set_ylabel('Predicted PM2.5 (\u03bcg/m\u00b3)')
axes[1].set_title(f'Periodic+RBF LOSO - Original Scale (R\u00b2 = {r2_orig:.3f})')
axes[1].set_xlim(0, max_val)
axes[1].set_ylim(0, max_val)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Site-level R\u00b2 distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# R\u00b2 histogram
valid_r2 = metrics_df['r2_log'].dropna()
axes[0].hist(valid_r2, bins=20, edgecolor='black')
axes[0].axvline(x=valid_r2.median(), color='r', linestyle='--', label=f'Median: {valid_r2.median():.3f}')
axes[0].set_xlabel('R\u00b2 (log scale)')
axes[0].set_ylabel('Number of sites')
axes[0].set_title('Site-level R\u00b2 Distribution')
axes[0].legend()

# RMSE vs number of observations
axes[1].scatter(metrics_df['n_obs'], metrics_df['rmse_log'], alpha=0.6)
axes[1].set_xlabel('Number of observations')
axes[1].set_ylabel('RMSE (log scale)')
axes[1].set_title('Site RMSE vs Sample Size')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Best and worst performing sites
print("Top 5 best-performing sites (highest R\u00b2):")
print(metrics_df.nlargest(5, 'r2_log')[['site', 'n_obs', 'rmse_log', 'r2_log']].to_string(index=False))

print("\nTop 5 worst-performing sites (lowest R\u00b2):")
print(metrics_df.nsmallest(5, 'r2_log')[['site', 'n_obs', 'rmse_log', 'r2_log']].to_string(index=False))

In [ ]:
# Map of site-level error on Montana map
site_coords = df_clean.groupby('ll_id')[['lat', 'lon']].first().reset_index()
site_with_error = site_coords.merge(metrics_df[['site', 'rmse_log', 'r2_log']],
                                     left_on='ll_id', right_on='site', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(16, 6),
                          subplot_kw={'projection': ccrs.PlateCarree()})

for ax in axes:
    ax.set_extent([-116.5, -104.0, 44.3, 49.1], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.STATES, edgecolor='black', linewidth=1.0)
    ax.add_feature(cfeature.LAND, facecolor='#f0f0f0')
    ax.add_feature(cfeature.LAKES, facecolor='lightblue', alpha=0.5)
    ax.add_feature(cfeature.RIVERS, edgecolor='lightblue', alpha=0.5)

# RMSE map
sc1 = axes[0].scatter(site_with_error['lon'], site_with_error['lat'],
                       c=site_with_error['rmse_log'], cmap='YlOrRd',
                       s=120, edgecolors='black', linewidth=0.8,
                       transform=ccrs.PlateCarree(), zorder=5)
plt.colorbar(sc1, ax=axes[0], label='RMSE (log scale)', shrink=0.7)
axes[0].set_title('Periodic+RBF LOSO - Site RMSE')
gl0 = axes[0].gridlines(draw_labels=True, alpha=0.3)
gl0.top_labels = False
gl0.right_labels = False

# R\u00b2 map
r2_min = min(0, site_with_error['r2_log'].min())
sc2 = axes[1].scatter(site_with_error['lon'], site_with_error['lat'],
                       c=site_with_error['r2_log'], cmap='RdYlGn',
                       s=120, edgecolors='black', linewidth=0.8,
                       vmin=r2_min, vmax=1,
                       transform=ccrs.PlateCarree(), zorder=5)
plt.colorbar(sc2, ax=axes[1], label='R\u00b2 (log scale)', shrink=0.7)
axes[1].set_title('Periodic+RBF LOSO - Site R\u00b2')
gl1 = axes[1].gridlines(draw_labels=True, alpha=0.3)
gl1.top_labels = False
gl1.right_labels = False

plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
residuals = pred_pm25 - actual_pm25

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].axvline(x=0, color='r', linestyle='--')
axes[0].set_xlabel('Residual (\u03bcg/m\u00b3)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Residual Distribution (mean={residuals.mean():.2f})')

axes[1].scatter(pred_pm25, residuals, alpha=0.2, s=5)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted PM2.5 (\u03bcg/m\u00b3)')
axes[1].set_ylabel('Residual (\u03bcg/m\u00b3)')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

## 5b. Daily Time Series at Best-Performing Site

In [ ]:
# Find the site with the lowest RMSE
best_site_row = metrics_df.loc[metrics_df['rmse_log'].idxmin()]
best_site = best_site_row['site']
print(f"Best site (lowest RMSE): {best_site}")
print(f"  RMSE (log): {best_site_row['rmse_log']:.4f}")
print(f"  R\u00b2 (log):   {best_site_row['r2_log']:.4f}")
print(f"  N obs:      {best_site_row['n_obs']:.0f}")

# Reconstruct per-site predictions by walking through sites in order
offset = 0
for site in sites:
    n = len(df_clean[df_clean['ll_id'] == site])
    if site == best_site:
        site_preds_log = all_predictions[offset:offset + n]
        site_actuals_log = all_actuals[offset:offset + n]
        break
    offset += n

# Convert to original scale
site_preds_orig = np.exp(site_preds_log) - 1
site_actuals_orig = np.exp(site_actuals_log) - 1

# Get dates and coordinates for this site
site_data = df_clean[df_clean['ll_id'] == best_site].sort_values('date')
site_dates = site_data['date'].values
site_lat = site_data['lat'].iloc[0]
site_lon = site_data['lon'].iloc[0]

# Sort by date
sort_idx = np.argsort(site_dates)
site_dates = site_dates[sort_idx]
site_actuals_orig = site_actuals_orig[sort_idx]
site_preds_orig = site_preds_orig[sort_idx]

# Compute original-scale metrics
site_rmse_orig = np.sqrt(np.mean((site_preds_orig - site_actuals_orig)**2))
site_mae_orig = np.mean(np.abs(site_preds_orig - site_actuals_orig))

# Small map showing site location
fig_map, ax_map = plt.subplots(figsize=(4, 3), subplot_kw={'projection': ccrs.PlateCarree()})
ax_map.set_extent([-116.5, -104.0, 44.3, 49.1], crs=ccrs.PlateCarree())
ax_map.add_feature(cfeature.STATES, edgecolor='black', linewidth=0.8)
ax_map.add_feature(cfeature.LAND, facecolor='#f0f0f0')
ax_map.plot(site_lon, site_lat, 'r*', markersize=14,
            transform=ccrs.PlateCarree(), zorder=5)
plt.tight_layout()
plt.show()

# Plot daily time series
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(site_dates, site_actuals_orig, label='Observed PM2.5', alpha=0.8, linewidth=1)
ax.plot(site_dates, site_preds_orig, label='Predicted PM2.5 (Periodic+RBF LOSO)', alpha=0.8, linewidth=1)
ax.set_xlabel('Date')
ax.set_ylabel('PM2.5 (\u03bcg/m\u00b3)')
ax.set_title(
    f'Daily PM2.5: Observed vs Predicted (Periodic+RBF LOSO)\n'
    f'Site {best_site} ({site_lat:.2f}\u00b0N, {abs(site_lon):.2f}\u00b0W) \u2014 '
    f'RMSE={site_rmse_orig:.2f} \u03bcg/m\u00b3, R\u00b2={best_site_row["r2_log"]:.3f}'
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Periodic Kernel Analysis

Examine the learned period across folds to verify the kernel is finding
a meaningful annual cycle.

In [ ]:
periods_df = pd.DataFrame(learned_periods)

print("Learned period (days) across LOSO folds:")
print(f"  Mean:   {periods_df['period_learned_days'].mean():.1f}")
print(f"  Median: {periods_df['period_learned_days'].median():.1f}")
print(f"  Std:    {periods_df['period_learned_days'].std():.1f}")
print(f"  Min:    {periods_df['period_learned_days'].min():.1f}")
print(f"  Max:    {periods_df['period_learned_days'].max():.1f}")
print(f"  Target: 365.25 (annual cycle)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of learned periods
axes[0].hist(periods_df['period_learned_days'], bins=15, edgecolor='black')
axes[0].axvline(x=365.25, color='r', linestyle='--', label='365.25 days (1 year)')
axes[0].set_xlabel('Learned Period (days)')
axes[0].set_ylabel('Number of folds')
axes[0].set_title('Learned Period Distribution')
axes[0].legend()

# Period vs fold R\u00b2
merged = periods_df.merge(metrics_df, on='site')
axes[1].scatter(merged['period_learned_days'], merged['r2_log'], alpha=0.6)
axes[1].axvline(x=365.25, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Learned Period (days)')
axes[1].set_ylabel('Fold R\u00b2 (log scale)')
axes[1].set_title('Learned Period vs Model Performance')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
import os

# Save site-level metrics
metrics_df['years'] = '2018-2019'
metrics_df['model'] = 'periodic_rbf'
if os.path.exists('../results/loso_site_metrics.csv'):
    existing = pd.read_csv('../results/loso_site_metrics.csv')
    pd.concat([existing, metrics_df], ignore_index=True).to_csv('../results/loso_site_metrics.csv', index=False)
else:
    metrics_df.to_csv('../results/loso_site_metrics.csv', index=False)
print("Site metrics saved to results/loso_site_metrics.csv")

# Save overall results
results = {
    'years': '2018-2019',
    'cv_method': 'LOSO',
    'model': 'periodic_rbf',
    'n_sites': n_sites,
    'n_predictions': len(all_predictions),
    'rmse_log': rmse_log,
    'mae_log': mae_log,
    'r2_log': r2_log,
    'rmse_orig': rmse_orig,
    'mae_orig': mae_orig,
    'r2_orig': r2_orig,
    'total_time_seconds': cv_total_time
}
new_row = pd.DataFrame([results])
if os.path.exists('../results/loso_overall_results.csv'):
    existing = pd.read_csv('../results/loso_overall_results.csv')
    pd.concat([existing, new_row], ignore_index=True).to_csv('../results/loso_overall_results.csv', index=False)
else:
    new_row.to_csv('../results/loso_overall_results.csv', index=False)
print("Overall results saved to results/loso_overall_results.csv")

# Save fold-level timing
timing_df = pd.DataFrame(fold_timings)
timing_df['years'] = '2018-2019'
timing_df['model'] = 'periodic_rbf'
if os.path.exists('../timings/loso_fold_timings.csv'):
    existing = pd.read_csv('../timings/loso_fold_timings.csv')
    pd.concat([existing, timing_df], ignore_index=True).to_csv('../timings/loso_fold_timings.csv', index=False)
else:
    timing_df.to_csv('../timings/loso_fold_timings.csv', index=False)
print("Fold timings saved to timings/loso_fold_timings.csv")

# Save learned periods
periods_df.to_csv('learned_periods.csv', index=False)
print("Learned periods saved to periodic/learned_periods.csv")

# Timing summary
print(f"\n{'='*50}")
print("Timing Summary")
print(f"{'='*50}")
print(f"Total CV time: {cv_total_time:.1f}s ({cv_total_time/60:.1f} min)")
print(f"\nPer-fold timing (across {len(timing_df)} folds):")
print(f"  Training:  mean={timing_df['train_time'].mean():.2f}s, std={timing_df['train_time'].std():.2f}s")
print(f"  Inference: mean={timing_df['infer_time'].mean():.3f}s, std={timing_df['infer_time'].std():.3f}s")
print(f"  Total:     mean={timing_df['fold_time'].mean():.2f}s, std={timing_df['fold_time'].std():.2f}s")

timing_log.summary()

## Summary

This notebook adds a locally periodic kernel component (Periodic × RBF on `day_of_year`)
to the baseline RBF+ARD kernel, following [Patel et al. (2022)](https://cdn.aaai.org/ojs/21467/21467-13-25480-1-2-20220628.pdf).

**Kernel:** `ScaleKernel(RBF_ARD(all 17 features)) + ScaleKernel(Periodic(day_of_year) × RBF(day_of_year))`

The periodic kernel is initialized with a period of 365.25 days (converted to
standardized units) and allowed to optimize. The learned period across folds
indicates whether the kernel recovers a meaningful annual cycle.

**Key difference from Patel et al.:** Their data is hourly urban measurements
with diurnal periodicity; ours is daily rural/semi-rural Montana data where
the dominant cycle is annual (winter inversions, summer wildfire smoke).